# UdaciMed | Notebook 3: Hardware Acceleration & Production Deployment

Welcome to the final phase of UdaciMed's optimization pipeline! In this notebook, you will implement cross-platform hardware acceleration techniques and strategize for the deployment of your optimized model across hardware targets.

## Recap: Optimization Journey

In [Notebook 2](02_architecture_optimization.ipynb), you have implemented architectural optimizations that brought you closer to your optimization targets.

Now, it is time to unlock further performance opportunities with hardware acceleration.

> **Your mission**: Transform your optimized model into a production-ready cross-platform deployment that meets production SLAs on this reference hardware, and finalize UdaciMed's deployment strategy across its diverse hardware fleet.

### Hardware acceleration

You will implement and evaluate **2 core deployment techniques\*** using [ONNX Runtime](https://onnxruntime.ai/):

1. **Mixed Precision (FP16)** - Utilizing 16-bit floating-point numbers to significantly speed up calculations and reduce memory usage on compatible hardware.
2. **Dynamic Batching** - Finding the best batch size to maximize throughput for offline tasks while maintaining low latency for real-time requests.

Additionally, you will analyze three deployment scenarios: GPU (TensorRT), CPU (OpenVINO), and Edge deployment considerations.

_\* Note that while you are expected to implement both deployment techniques, you can decide whether to keep either or both in your final deployment strategy to best achieve targets._

---

Through this notebook, you will:

- **Convert PyTorch model to ONNX** for cross-platform deployment
- **Apply hardware acceleration using ONNX Runtime** on the reference T4 device
- **Benchmark end-to-end performance** against SLAs
- **Validate clinical safety** across the deployment pipeline
- **Analyze alternative deployment strategies** for diverse hardware environments

**Let's deliver a production-ready, hardware-accelerated diagnostic deployment!**

## Step 1: Setup the environment

First, let's set up the environment and understand our reference hardware capabilities. 

This ensures our optimization and benchmarking code will run smoothly.

In [3]:
! pip uninstall -y onnxruntime onnxruntime-gpu
! pip install onnxruntime-gpu

Found existing installation: onnxruntime 1.22.1
Uninstalling onnxruntime-1.22.1:
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/bin/onnxruntime_test' -> '/tmp/pip-uninstall-yhamzzgj/onnxruntime_test'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 106, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", line 722, in uninstall
    uninstalled_pathset.remove(auto_confirm, verbose)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_uninstall.py", line 370, 

In [4]:
import onnxruntime as ort
print(ort.get_available_providers())

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [5]:
# Make sure that libraries are dynamically re-loaded if changed
%load_ext autoreload
%autoreload 2

In [6]:
# Import core libraries
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
import pickle
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Literal
import warnings
warnings.filterwarnings('ignore')

# Import project utilities
from utils.data_loader import (
    load_pneumoniamnist,
    get_sample_batch
)
from utils.model import (
    create_baseline_model,
    get_model_info
)
from utils.evaluation import (
    evaluate_with_multiple_thresholds
)
from utils.profiling import (
    PerformanceProfiler,
    measure_time
)
from utils.visualization import (
    plot_performance_profile,
    plot_batch_size_comparison
)
from utils.architecture_optimization import (
    create_optimized_model
)

In [7]:
# Set device and analyze hardware capabilities
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Check tensor core support for mixed precision - crucial for FP16 acceleration
    gpu_compute = torch.cuda.get_device_properties(0).major
    tensor_core_support = gpu_compute >= 7  # Volta+ architecture
    print(f"Tensor Core Support: {tensor_core_support}")
else:
    print("WARNING: CUDA not available - hardware acceleration will be limited")

print("Default hardware acceleration environment ready!")

# Verify ONNX Runtime GPU support
print(f"\nONNX Runtime available providers: {ort.get_available_providers()}")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB
Tensor Core Support: True
Default hardware acceleration environment ready!

ONNX Runtime available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


> **Getting ready for acceleration**: The checks above highlight two critical facts for our mission:
> 1. Our reference hardware has tensor core support, which can dramatically speed up 16-bit floating-point (FP16) calculations; for other hardware deployments, like CPUs that lack this feature, we would need to rely on different techniques (such as 8-bit integer quantization (INT8)) to achieve similar acceleration.
> 2. ONNX Runtime providers are available for our primary targets: CUDAExecutionProvider for GPU and CPUExecutionProvider for CPU. This allows us to benchmark on both platforms. For a true mobile or edge deployment, we would need to use a specialized package like ONNX Runtime Mobile, which is built separately to keep the application lightweight.
> 
> Our task is to meet SLAs on our current device, which means we must **_benchmark against the GPU_** to see if we've met our goals.

## Step 2: Load test data and optimized model with configuration

The model is needed for deployment, and the optimization results for comparison.

Test data is needed for both conversion and final performance testing.

In [8]:
# Define dataset loading parameters
img_size = 64
batch_size = 32

# Load test dataset for final evaluation
test_loader = load_pneumoniamnist(
    split="test", 
    download=True, 
    size=img_size,
    batch_size=batch_size,
    subset_size=None
)

# Get sample batch for profiling
sample_images, sample_labels = get_sample_batch(test_loader)
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)

print(f"Test data loaded: {sample_images.shape} batch for hardware acceleration profiling")

Using downloaded and verified file: /voc/work/.medmnist/pneumoniamnist_64.npz
Test data loaded: torch.Size([32, 3, 64, 64]) batch for hardware acceleration profiling


> **Batch size strategy**: Your batch size choice impacts memory usage, latency, and throughput. 
> 
> Consider: What batch size best applied for each deployment scenario? Don't forget to review the batch analysis plot from Notebook 2!

In [9]:
# Load optimized model and results from notebook 2

# Resolve experiment name from available result artifacts (latest file wins).
results_dir = Path("../results")
result_files = sorted(results_dir.glob("optimization_results_*.pkl"), key=lambda p: p.stat().st_mtime)
if not result_files:
    raise FileNotFoundError(
        "No optimization results found. Run Notebook 2 first to generate optimization_results_*.pkl"
    )

latest_results_path = result_files[-1]
experiment_name = latest_results_path.stem.replace("optimization_results_", "")

with open(latest_results_path, "rb") as f:
    optimization_results = pickle.load(f)

print("Loaded optimization results from Notebook 2:")
print(f"   Experiment: {experiment_name}")
print(f"   Model: {optimization_results['model_name']}")
print(f"   Clinical Performance: {optimization_results['clinical_performance']['optimized']['sensitivity']:.1%} sensitivity")
print(f"   Architecture Speedup: {optimization_results['performance_improvements']['latency_speedup']:.2f}x")
print(f"   Memory Reduction: {optimization_results['performance_improvements']['memory_reduction_percent']:.1f}%")

Loaded optimization results from Notebook 2:
   Experiment: optimized_interpolation_removal_depthwise_separable_channel_optimization_use_amp
   Model: ResNet-18 Optimized
   Clinical Performance: 99.0% sensitivity
   Architecture Speedup: 0.60x
   Memory Reduction: 80.2%


> **HINT: Finding your optimization results**
> 
> Your optimization results from Notebook 2 should be saved as:
> - Results file: `../results/optimization_results_{experiment_name}.pkl`
> - Model weights: `../results/optimized_model.pth`
> 
> The experiment name typically combines your optimization techniques, like:
> - `"interpolation-removal_depthwise-separable"`
> - `"channel-reduction_grouped-conv"`

In [10]:
# Get the optimization configuration
opt_config = optimization_results['optimization_config']
optimized_model = None

# Rebuild the architecture exactly as in notebook 2, then load trained optimized weights.
baseline_for_load = create_baseline_model(
    num_classes=2,
    input_size=img_size,
    pretrained=False,
    fine_tune=True,
)
optimized_model = create_optimized_model(baseline_for_load, opt_config)

weights_path = Path("../results/optimized_model.pth")
if not weights_path.exists():
    raise FileNotFoundError(
        f"Optimized weights not found at {weights_path}. Run Notebook 2 save step first."
    )

state_dict = torch.load(weights_path, map_location=device)
optimized_model.load_state_dict(state_dict, strict=False)
optimized_model = optimized_model.to(device)
optimized_model.eval()

print("Optimized model loaded and ready for ONNX export")
print(f"   Weights path: {weights_path}")
print(f"   Device: {device}")

Starting clinical model optimization pipeline...
   Applying interpolation removal optimization...
Applying native resolution optimization (64x64)...
INTERPOLATION REMOVAL completed.
   Applying depthwise separable optimization...
Applying depthwise separable convolution optimization...
DEPTHWISE SEPARABLE completed: Successfully applied to layers with 16 replacements
   Applying channel optimization optimization...
Applying channel-level hardware optimizations...
   Converted 0 activation(s) to in-place
   Converted model to channels_last memory format
CHANNEL OPTIMIZATION completed
Applied optimizations in order: interpolation_removal → depthwise_separable → channel_optimization
Optimized model loaded and ready for ONNX export
   Weights path: ../results/optimized_model.pth
   Device: cuda


## Step 3: Convert model with hardware acceleration for production deployment

Convert the optimized model to [ONNX (Open Neural Network Exchange)](https://onnx.ai/) with optional hardware accelerations. 

**IMPORTANT**: You are tasked to implement both hardware optimizations even if you decide to disable them for the final export.

In [11]:
# Define deployment configuration for ONNX export.
# FP16 is enabled only when CUDA is available; dynamic batching helps support variable request sizes.
use_fp16 = torch.cuda.is_available()
use_dynamic_batching = True

print("Deployment export configuration:")
print(f"   FP16 enabled: {use_fp16}")
print(f"   Dynamic batching: {use_dynamic_batching}")

Deployment export configuration:
   FP16 enabled: True
   Dynamic batching: True


In [12]:
# Convert PyTorch model to ONNX format (for cross-platform deployment)

def export_model_to_onnx(model: nn.Module, input_tensor: torch.Tensor,
                        export_path: str, model_name: str = "pneumonia_detection",
                        fp16_mode: bool = use_fp16, dynamic_batching: bool = use_dynamic_batching) -> str:
    """
    Export PyTorch model to ONNX format for production deployment.
    Apply hardware optimizations if selected.

    Args:
        model: PyTorch model to export
        input_tensor: Sample input tensor for shape inference
        export_path: Directory to save the ONNX model
        model_name: Name for the exported ONNX file
        fp16_mode: If True, exports the model in FP16 (mixed precision)
        dynamic_batching: If True, configures the model to accept variable batch sizes

    Returns:
        Path to exported ONNX model
    """
    # Define output path, and ensure it exists
    onnx_path = f"{export_path}/{model_name}.onnx"
    Path(export_path).mkdir(parents=True, exist_ok=True)

    # Keep export dtype consistent between weights and example input.
    model.eval()
    export_input = input_tensor
    export_model = model.to(input_tensor.device)

    if fp16_mode:
        if not torch.cuda.is_available():
            print("WARNING: FP16 requested but CUDA unavailable; falling back to FP32 export")
            fp16_mode = False
        else:
            export_model = export_model.half()
            export_input = export_input.half()
    else:
        export_model = export_model.float()
        export_input = export_input.float()

    print(f"Exporting model to ONNX format...")
    print(f"   Input shape: {export_input.shape}")
    print(f"   Input dtype: {export_input.dtype}")
    print(f"   FP16 mode: {fp16_mode}")
    print(f"   Dynamic batching: {dynamic_batching}")
    print(f"   Export path: {onnx_path}")

    dynamic_axes = None
    if dynamic_batching:
        dynamic_axes = {
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        }

    # Export to ONNX format with defined parameters
    torch.onnx.export(
        export_model,
        export_input,
        onnx_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes=dynamic_axes,
        opset_version=16,
        do_constant_folding=True,
        verbose=False,
    )

    print(f"ONNX export completed: {onnx_path}")

    # Verify ONNX model integrity - sanity check
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print("   ONNX model verification passed")
    except Exception as e:
        print(f"   WARNING: ONNX verification failed: {str(e)}")

    # Restore model precision for potential downstream PyTorch use.
    if fp16_mode:
        model.float()

    return onnx_path

# Export the model to ONNX
onnx_model_path = export_model_to_onnx(
    model=optimized_model,
    input_tensor=sample_images,
    export_path="../results/onnx_models",
    model_name="udacimed_pneumonia_optimized",
)
print(f"Final ONNX artifact: {onnx_model_path}")

Exporting model to ONNX format...
   Input shape: torch.Size([32, 3, 64, 64])
   Input dtype: torch.float16
   FP16 mode: True
   Dynamic batching: True
   Export path: ../results/onnx_models/udacimed_pneumonia_optimized.onnx


ONNX export completed: ../results/onnx_models/udacimed_pneumonia_optimized.onnx
   ONNX model verification passed
Final ONNX artifact: ../results/onnx_models/udacimed_pneumonia_optimized.onnx


## Step 4: Deploy with ONNX Runtime

With our model saved in the ONNX format, we can now load it into the [ONNX Runtime (ORT)](https://onnxruntime.ai/getting-started). 

ORT is a high-performance inference engine that can execute models on different hardware backends through its **Execution Providers (EPs)**. 

In [13]:
# This function creates an ONNX Runtime Inference Session.

# Prefer GPU when available to target production acceleration path.
use_gpu = torch.cuda.is_available()

def create_inference_session(model_path: str, use_gpu: bool = use_gpu) -> ort.InferenceSession:
    """
    Creates an ONNX Runtime inference session.

    Args:
        model_path: Path to the ONNX model file.
        use_gpu: If True, configures the session to use the CUDA Execution Provider.

    Returns:
        An ONNX Runtime InferenceSession object.
    """
    print(f"Creating ONNX Runtime session for {'GPU' if use_gpu else 'CPU'}...")

    available_providers = ort.get_available_providers()
    providers = ["CPUExecutionProvider"]
    if use_gpu and "CUDAExecutionProvider" in available_providers:
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]

    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    # Create the ONNX Runtime InferenceSession
    session = ort.InferenceSession(
        model_path,
        sess_options=session_options,
        providers=providers,
    )

    print(f"Session created with providers: {session.get_providers()}")
    return session

# Create the session for our exported ONNX model.
inference_session = create_inference_session(onnx_model_path)

Creating ONNX Runtime session for GPU...
Session created with providers: ['CPUExecutionProvider']


2026-10-05 05:43:01.927381372 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcudnn.so.9: cannot open shared object file: No such file or directory

2026-10-05 05:43:01.927413495 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.


# Step 5: Benchmark model performance on all metrics

Now that we have a hardware-accelerated inference session, it's time to measure its performance. 

Unlike a server-based approach, we will perform direct, client-side benchmarking. This gives us precise measurements of the model's raw inference speed and resource consumption on our target hardware.

In [14]:
# Define a helper function to get input details and type

def get_input_details(session: ort.InferenceSession) -> Tuple[str, Tuple, np.dtype]:
    """
    Gets the input name, shape, and dtype for an ONNX Runtime session.
    """
    input_details = session.get_inputs()[0]
    input_name = input_details.name

    # Match NumPy dtype to ONNX input tensor dtype.
    input_type = str(input_details.type).lower()
    is_fp16 = "float16" in input_type

    # Determine the correct numpy dtype
    input_dtype = np.float16 if is_fp16 else np.float32

    return input_name, input_details.shape, input_dtype

In [15]:
# This is the main benchmarking function.

def benchmark_performance(session: ort.InferenceSession,
                          test_data: torch.Tensor,
                          batch_sizes: List[int],
                          num_runs: int = 50) -> Dict[str, Any]:
    """
    Benchmarks the performance of an ONNX Runtime session.

    Args:
        session: The ONNX Runtime inference session.
        test_data: A batch of test data for inference.
        batch_sizes: A list of batch sizes to test.
        num_runs: The number of inference runs to average for timing.

    Returns:
        A dictionary containing the performance results for each batch size.
    """
    results = {}
    output_name = session.get_outputs()[0].name

    input_name, _, input_dtype = get_input_details(session)
    print(f"Benchmarking with input dtype: {input_dtype}")

    for batch_size in batch_sizes:
        print(f"--- Benchmarking Batch Size: {batch_size} ---")

        # Prepare batch data
        input_array = test_data[:batch_size].detach().cpu().numpy().astype(input_dtype)

        # Warm-up runs to stabilize GPU clocks and cache
        for _ in range(10):
            session.run([output_name], {input_name: input_array})

        # Timed runs
        latencies = []

        # Perform the timed inference runs
        for _ in range(num_runs):
            start_time = time.perf_counter()
            session.run([output_name], {input_name: input_array})
            end_time = time.perf_counter()
            latencies.append((end_time - start_time) * 1000)  # Convert to ms

        # Measure peak GPU memory usage (if CUDA is active)
        if torch.cuda.is_available() and "CUDAExecutionProvider" in session.get_providers():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            session.run([output_name], {input_name: input_array})
            peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
        else:
            peak_memory_mb = 0  # No GPU memory to measure on CPU

        # Calculate metrics
        avg_latency_ms = float(np.mean(latencies))
        throughput_sps = (batch_size / avg_latency_ms) * 1000  # Samples per second

        results[batch_size] = {
            "avg_latency_ms": avg_latency_ms,
            "throughput_sps": throughput_sps,
            "peak_memory_mb": float(peak_memory_mb),
        }
        print(f"  Avg Latency: {avg_latency_ms:.3f} ms")
        print(f"  Throughput: {throughput_sps:,.2f} samples/sec")
        print(f"  Peak GPU Memory: {peak_memory_mb:.2f} MB")

    return results

# Test both real-time (BS=1) and throughput-oriented batch sizes.
batch_sizes_to_test = [1, 8, 16, 32]

# Run the benchmark
benchmark_results = benchmark_performance(
    session=inference_session,
    test_data=sample_images,
    batch_sizes=batch_sizes_to_test,
)

Benchmarking with input dtype: <class 'numpy.float16'>
--- Benchmarking Batch Size: 1 ---
  Avg Latency: 0.828 ms
  Throughput: 1,208.32 samples/sec
  Peak GPU Memory: 0.00 MB
--- Benchmarking Batch Size: 8 ---
  Avg Latency: 4.314 ms
  Throughput: 1,854.43 samples/sec
  Peak GPU Memory: 0.00 MB
--- Benchmarking Batch Size: 16 ---
  Avg Latency: 8.111 ms
  Throughput: 1,972.72 samples/sec
  Peak GPU Memory: 0.00 MB
--- Benchmarking Batch Size: 32 ---
  Avg Latency: 9.544 ms
  Throughput: 3,352.88 samples/sec
  Peak GPU Memory: 0.00 MB


## Step 6: Assess if production targets are met

Final evaluation against all production deployment requirements. Meeting all targets demonstrates successful optimization for UdaciMed's deployment requirements.

In [16]:
# Define production targets
# Note that we are skipping FLOP analysis here because not directly impacted by hardware acceleration
PRODUCTION_TARGETS = {
    'memory': 100,               # MB - Achievable with mixed precision
    'throughput': 2000,          # samples/sec - Target for multi-tenant deployment
    'latency': 3,                # ms - Individual inference time for real-time scenarios
    'sensitivity': 98,           # % - Clinical safety requirement (non-negotiable)
}

In [17]:
# STEP 1: Extract the best batch configuration from the benchmark results

# Initialize variables to hold the best results found.
latency_for_target = float('inf')
max_throughput = 0
best_throughput_bs = None
memory_at_max_throughput = 0

# Check if the real-time latency scenario (batch size 1) was tested.
if 1 in benchmark_results:
    latency_for_target = benchmark_results[1]['avg_latency_ms']
else:
    print("WARNING: Batch size 1 not found in results. Real-time latency target cannot be evaluated.")

# Find the batch size that yielded the highest throughput.
if benchmark_results:
    best_throughput_bs = max(benchmark_results, key=lambda bs: benchmark_results[bs]['throughput_sps'])
    max_throughput = benchmark_results[best_throughput_bs]['throughput_sps']
    memory_at_max_throughput = benchmark_results[best_throughput_bs]['peak_memory_mb']

# Get model file size as another memory metric
model_file_size_mb = Path(onnx_model_path).stat().st_size / (1024 * 1024)

print("\n--- Performance Analysis ---")
print(f"Real-time Latency (BS=1): {f'{latency_for_target:.3f} ms' if latency_for_target != float('inf') else 'Not Tested'}")
if best_throughput_bs is not None:
    print(f"Max Throughput: {max_throughput:,.2f} samples/sec (at Batch Size={best_throughput_bs})")
    print(f"Peak GPU memory at max throughput: {memory_at_max_throughput:.2f} MB")
print(f"Model file size: {model_file_size_mb:.2f} MB")


--- Performance Analysis ---
Real-time Latency (BS=1): 0.828 ms
Max Throughput: 3,352.88 samples/sec (at Batch Size=32)
Peak GPU memory at max throughput: 0.00 MB
Model file size: 2.77 MB


In [18]:
# STEP 2: Define a function to validate the clinical performance using the ONNX session.

def validate_clinical_performance(session: ort.InferenceSession,
                                  test_loader,
                                  threshold: float = 0.5) -> Dict[str, Any]:
    """
    Validates clinical performance (sensitivity) using the ONNX Runtime session.
    """
    print("\nValidating clinical performance on test data...")
    input_name, _, input_dtype = get_input_details(session)
    output_name = session.get_outputs()[0].name

    all_predictions = []
    all_labels = []

    for batch_inputs, batch_labels in test_loader:
        # Prepare input
        input_array = batch_inputs.detach().cpu().numpy().astype(input_dtype)

        # Run inference
        results = session.run([output_name], {input_name: input_array})
        logits = torch.from_numpy(results[0])

        # Process output
        probabilities = torch.softmax(logits, dim=1)[:, 1]  # Probability of class 1 (pneumonia)
        all_predictions.extend(probabilities.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

    # Calculate metrics
    predictions = np.array(all_predictions)
    labels = np.array(all_labels).flatten()
    pred_classes = (predictions > threshold).astype(int)

    tp = np.sum((pred_classes == 1) & (labels == 1))
    fn = np.sum((pred_classes == 0) & (labels == 1))

    sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"Clinical validation completed on {len(labels)} samples.")
    print(f"  Calculated Sensitivity: {sensitivity:.2f}% (at threshold={threshold})")

    return {"sensitivity": sensitivity}


# Reuse Notebook 2's selected threshold when available; default to 0.4.
clinical_threshold = 0.4
threshold_candidates = [
    optimization_results.get("selected_threshold"),
    optimization_results.get("clinical_performance", {}).get("selected_threshold"),
]
for candidate in threshold_candidates:
    if isinstance(candidate, (int, float)) and 0.0 < float(candidate) < 1.0:
        clinical_threshold = float(candidate)
        break

clinical_results = validate_clinical_performance(
    session=inference_session,
    test_loader=test_loader,
    threshold=clinical_threshold,
)


Validating clinical performance on test data...


Clinical validation completed on 624 samples.
  Calculated Sensitivity: 98.97% (at threshold=0.4)


In [19]:
# Manually set/check FLOP reduction target from Notebook 2 architecture results.
flops_target_reduction = 80
flops_achieved_reduction = optimization_results['performance_improvements']['flop_reduction_percent']
flp_ok = flops_achieved_reduction >= flops_target_reduction

# Check if targets are met
mem_ok = model_file_size_mb < PRODUCTION_TARGETS['memory']
lat_ok = latency_for_target < PRODUCTION_TARGETS['latency']
thr_ok = max_throughput > PRODUCTION_TARGETS['throughput']
sen_ok = clinical_results['sensitivity'] > PRODUCTION_TARGETS['sensitivity']
all_ok = all([mem_ok, lat_ok, thr_ok, sen_ok, flp_ok])

print(f"| Metric          | Target                    | Achieved                  | Status  |")
print(f"|-----------------|---------------------------|---------------------------|---------|")
print(f"| Memory          | < {PRODUCTION_TARGETS['memory']} MB                  | {model_file_size_mb:.2f} MB                   | {'✔️ Met' if mem_ok else '✖️ Missed'}  |")
print(f"| Latency         | < {PRODUCTION_TARGETS['latency']} ms                    | {latency_for_target:.3f} ms                  | {'✔️ Met' if lat_ok else '✖️ Missed'}  |")
print(f"| Throughput      | > {PRODUCTION_TARGETS['throughput']:,} samples/sec       | {max_throughput:,.2f} samples/sec     | {'✔️ Met' if thr_ok else '✖️ Missed'}  |")
print(f"| FLOP Reduction  | > {flops_target_reduction}%                     | {flops_achieved_reduction:.1f}%                     | {'✔️ Met' if flp_ok else '✖️ Missed'}  |")
print(f"| Sensitivity     | > {PRODUCTION_TARGETS['sensitivity']}%                     | {clinical_results['sensitivity']:.2f}%                    | {'✔️ Met' if sen_ok else '✖️ Missed'}  |")
print(f"\nOverall Result: {'CONGRATS: All production targets met!' if all_ok else 'WARNING: Some targets were not met. Further optimization may be needed.'}")
print("\nNOTE: FLOPs are inherited from Notebook 2 architecture changes; deployment acceleration affects latency/throughput/runtime memory.")

| Metric          | Target                    | Achieved                  | Status  |
|-----------------|---------------------------|---------------------------|---------|
| Memory          | < 100 MB                  | 2.77 MB                   | ✔️ Met  |
| Latency         | < 3 ms                    | 0.828 ms                  | ✔️ Met  |
| Throughput      | > 2,000 samples/sec       | 3,352.88 samples/sec     | ✔️ Met  |
| FLOP Reduction  | > 80%                     | 98.5%                     | ✔️ Met  |
| Sensitivity     | > 98%                     | 98.97%                    | ✔️ Met  |

Overall Result: CONGRATS: All production targets met!

NOTE: FLOPs are inherited from Notebook 2 architecture changes; deployment acceleration affects latency/throughput/runtime memory.


---

## Step 7: Cross-platform deployment analysis

We have successfully optimized our model to meet _UdaciMed's Universal Performance Standard_ on our standardized target device. 

With ONNX, we can easily deploy this optimized model across UdaciMed's diverse hardware fleet just by [changing the Execution Providers](https://onnxruntime.ai/docs/execution-providers/):

| Deployment Target	| Recommended Technology |	Primary Goal	 |	Key Trade-Off | 
| :--- | :--- | :--- | :--- |
| GPU Server (Cloud/On-Prem) |		ONNX Runtime + TensorRT		 |Max Throughput 	 |	Highest performance vs. more complex setup. | 
| CPU Workstation (Hospital) |		ONNX Runtime + OpenVINO		 |Low Latency  |		Excellent CPU speed vs. being tied to Intel hardware. | 
| Mobile/Edge Device (Clinic) |		ONNX Runtime Mobile		 | Small Footprint  |		Maximum portability vs. reduced model precision (quantization). | 

But **what if we need to squeeze out every last drop of performance from each deployment target?** To do this, let's consider moving beyond the portable ONNX format and use specialized, hardware-specific frameworks.

### **Step 7.1: Optimization strategy for specialized GPU server deployment**

The saved run reports **0.828 ms batch-1 latency**, **3,352.88 samples/sec maximum throughput**, and **98.97% sensitivity at threshold 0.4**. However, CUDA initialization failed because `libcudnn.so.9` was missing, and the session used `CPUExecutionProvider`. These are CPU fallback measurements, not a measured GPU baseline. The 2.77 MB memory figure is ONNX artifact size; runtime peak memory was not established.

#### GPU deployment options

Expected performance depends on GPU, operators, model precision, and request concurrency. No CUDA, TensorRT, or Triton performance was measured in this run.

| Approach | How it Works | Key Performance Contributor | Complexity/Overhead | UdaciMed Suitability |
| :--- | :--- | :--- | :--- | :--- |
| **ONNX Runtime with CUDA EP** | Runs supported ONNX operators through CUDA libraries, with fallback for unsupported nodes. | Direct GPU kernels, graph optimizations, and FP16 on compatible hardware. | Simple app integration, but requires compatible CUDA/cuDNN libraries. | First GPU validation path; initialization failed in the saved run because cuDNN 9 was unavailable. |
| **ONNX Runtime with TensorRT EP** | Builds TensorRT engines for supported graph partitions; other nodes may run on CUDA/CPU providers. | Engine optimization, fusion, and kernel selection can improve performance. | Engine build/cache management, shape profiles, and NVIDIA version/hardware compatibility add operational work. | Evaluate only after the CUDA baseline works; require per-shape performance and clinical validation. |
| **Triton Inference Server with TensorRT backend** | Hosts TensorRT engines behind an inference server with model management and request scheduling. | Dynamic batching and concurrent model instances can increase aggregate GPU utilization. | Highest operational footprint: server/container, networking, monitoring, and queue management. | Candidate for centralized multi-tenant serving; likely excessive for one clinic workstation. |

**1. What is the main business risk of choosing TensorRT over CUDA EP?**

TensorRT increases dependence on NVIDIA hardware, supported operators, and compatible runtime/engine versions. Hardware or driver upgrades may require rebuilding engines and repeating performance and clinical validation, adding maintenance cost and risk to fleet portability.

**2. Why might a small clinic not want Triton?**

A single workstation may have too few concurrent requests to benefit from server-side batching or model management. Running and maintaining Triton adds networking, monitoring, deployment, and queueing overhead compared with direct ONNX Runtime integration.

#### Strategic choice

**Recommendation for the long-term multi-tenant GPU service:** Evaluate Triton with a TensorRT backend for centralized model management and concurrent-request scheduling, but adopt it only if load testing demonstrates a benefit over direct CUDA EP that justifies its operational cost. First resolve the saved run's CUDA 12/cuDNN 9 dependency error, establish a real CUDA EP baseline, measure runtime memory, and revalidate sensitivity after conversion; current 5/5 output is from CPU fallback and uses artifact size for its memory gate.

#### Example Triton configuration for FP16 and dynamic batching

This is a proposed configuration for the existing FP16 ONNX model with Triton's ONNX Runtime backend. It is not a TensorRT configuration and is not benchmarked by this notebook. The tested batch sizes were 1, 8, 16, and 32, so the example caps batches at 32.

```protobuf
name: "udacimed_pneumonia_prod"
platform: "onnxruntime_onnx"
max_batch_size: 32

input [
  {
    name: "input"
    data_type: TYPE_FP16
    dims: [ 3, 64, 64 ]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP16
    dims: [ 2 ]
  }
]

instance_group [
  {
    count: 1
    kind: KIND_GPU
  }
]

dynamic_batching {
  preferred_batch_size: [ 8, 16, 32 ]
  max_queue_delay_microseconds: 1000
}
```

`TYPE_FP16` must match the exported model's input/output types; editing `config.pbtxt` does not convert an FP32 model or install CUDA/cuDNN. Triton's dynamic batcher combines independent incoming requests up to the configured size and queue delay; unlike ONNX dynamic axes, which only permit variable-sized batches, Triton batching schedules requests together. Validate end-to-end latency including queue and network time, as well as throughput and sensitivity under realistic concurrent load.

For a TensorRT backend trial, build an engine with optimization profiles covering batch sizes 1 through 32, store it in the Triton model repository, and use the TensorRT platform/backend with configuration matching the engine's I/O. Do not infer TensorRT or Triton SLAs from the current CPU fallback measurements.

References: [Triton model configuration](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_configuration.html) and [dynamic batching](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html).

### **Step 7.2: Optimization strategy for specialized CPU deployment**

CPU deployment matters because many hospitals and clinics have no dedicated GPU. The CPU should provide predictable single-patient inference with limited memory and preserve clinical sensitivity. FP16 is generally not a useful CPU speed optimization; FP32 is the conservative starting point, while INT8 may improve speed on supported CPUs but requires calibration and clinical revalidation.

#### CPU deployment options

These are qualitative expectations, not measurements from this notebook. Actual results depend on processor, runtime build, thread settings, operators, and concurrent load.

| Approach | How it Works | Conversion Path | Memory Footprint | Performance | UdaciMed Suitability |
|----------|--------------|-----------------|------------------|-------------|----------------------|
| **PyTorch on CPU** | Runs the model using PyTorch CPU operators. | Load PyTorch checkpoint directly. | Highest framework/process overhead among these options. | Useful correctness baseline; often slower than optimized inference runtimes. | Reference/debug option, not preferred production runtime. |
| **ONNX Runtime with Default CPU** | Executes the ONNX graph with CPU graph optimizations and available kernels. | Export to ONNX; no separate IR conversion required. | Moderate; measure process RSS and peak inference memory. | Portable baseline; fusion and optimized kernels can improve CPU performance. | Strong default for mixed-vendor fleets and simple integration. |
| **ONNX Runtime with OpenVINO EP** | Delegates supported graph nodes to OpenVINO while retaining the ONNX Runtime interface. | Keep ONNX as interface; install compatible OpenVINO provider/runtime. | Moderate; depends on graph partitioning. | May accelerate supported operators on Intel hardware; benchmark against default CPU EP. | Practical Intel-specific option when already using ONNX Runtime. |
| **Native OpenVINO Runtime (IR)** | Compiles and executes a model in OpenVINO's native representation. | Convert ONNX/model to OpenVINO IR and compile for CPU. | Potentially lean; verify process and peak memory empirically. | Device-specific graph optimization may help; benefit is CPU/model-dependent. | Maintain a separate artifact only when measured Intel gains justify it. |
| **OpenVINO backend for Triton** | Serves OpenVINO models through Triton with centralized scheduling and model management. | Convert/package an OpenVINO-supported model in Triton's model repository. | Highest service overhead; profile server and model together. | Concurrent requests and batching may increase aggregate utilization with queueing trade-offs. | Useful for centralized multi-model/multi-department service, not a single workstation. |

**1. What is the advantage of Native OpenVINO IR, and when is conversion worthwhile?**

Native IR lets OpenVINO compile and optimize the graph for its target device and can expose transformations or precision paths unavailable through a generic ONNX execution route. Maintain the extra artifact only when representative Intel hardware benchmarks show a meaningful benefit and clinical validation confirms prediction parity.

**2. When does Triton make sense despite its memory overhead?**

Triton is appropriate when a central service shares resources across many concurrent requests, departments, or model versions and centralized scheduling/operations justify the extra memory and maintenance. It is usually unnecessary for a single workstation serving one user.

**3. What metric must be revalidated for clinical safety?**

Revalidate sensitivity at the locked operating threshold on a representative held-out dataset after every runtime, precision, or model conversion. Compare predictions/logits and uncertainty where feasible; unchanged aggregate accuracy alone does not establish safety.

#### Strategic choice

**Recommendation for a typical hospital workstation:** Start with ONNX Runtime CPU EP in FP32 for a simple cross-vendor deployment and benchmark it on the actual workstation. Consider OpenVINO EP or native IR only for Intel hosts where measurements show material improvement without reducing validated sensitivity.

#### Example OpenVINO hospital configuration

This is a conservative starting profile, not a measured or certified production configuration. Export a separate FP32 ONNX artifact for this CPU path; do not label the CUDA-oriented FP16 artifact as FP32. Tune thread count and memory using representative workstation load tests.

```yaml
# openvino_hospital_config.yaml
# UdaciMed hospital workstation deployment

model_optimization:
  input_model: "udacimed_pneumonia_optimized_fp32.onnx"
  target_device: "CPU"
  precision: "FP32"
  optimization_level: "ACCURACY"
  quantization:
    enabled: false
    calibration_dataset_size: 0

deployment_config:
  cpu_threads: 4
  memory_pool_mb: 256
  max_batch_size: 1
  inference_timeout_ms: 500

clinical_validation:
  sensitivity_threshold: 98.0
  validation_dataset_size: 624
  comparison_baseline: "ONNXRuntime_CPU_EP_FP32"
```

**Configuration rationale:**

- **FP32:** Conservative initial CPU precision; avoids an unvalidated quantization trade-off.
- **ACCURACY:** Prioritizes numerical behavior before performance tuning.
- **Quantization disabled / zero calibration samples:** INT8 is not enabled until representative calibration data and clinical results are available.
- **Four CPU threads:** Starting point that leaves cores for other hospital software; tune to host capacity and concurrency.
- **256 MB memory pool:** Provisional per-instance budget; confirm process RSS and peak memory under load.
- **Batch size one:** Matches interactive single-patient use and avoids batching queue delay.
- **500 ms timeout:** Example service-level timeout, not the model latency gate; enforce and measure the <3 ms inference requirement separately.
- **98% sensitivity:** Clinical floor; acceptance requires measured sensitivity strictly above the notebook's >98% gate.
- **624 validation cases:** Matches the current test-set size for comparability; use a locked, representative cohort and report uncertainty before clinical release.
- **CPU EP baseline:** Compare alternative runtimes with identical input, threshold, dataset, and timing protocol; this notebook has no validated GPU/Triton baseline.

### **Step 7.3: Optimization strategy for mobile and edge deployment**

UdaciMed's mobile strategy should prioritize reliable offline screening, manageable app size and battery use, broad access across lower-cost devices, and a controlled clinical update process. The table describes platform capabilities qualitatively; this notebook has not benchmarked or clinically validated a mobile conversion.

#### Mobile deployment options

| Platform | How it Works | Key Strength | Main Trade-Off | UdaciMed Suitability |
|----------|----------------|------------|---------------|-------------------|
| **ONNX Runtime Mobile** | Runs a pruned ONNX Runtime package and ONNX model locally on supported mobile platforms. | One model format and a shared cross-platform integration path; supports offline inference. | Larger runtime or less device-specific acceleration than a narrowly targeted native runtime; supported operators must be checked. | Best first cross-platform prototype when development capacity is limited and broad Android/iOS reach matters. |
| **ExecuTorch** | Exports a PyTorch model to an edge program and runs it through a lightweight runtime with supported delegates/backends. | Keeps a PyTorch-oriented export workflow and allows backend-specific acceleration. | Export/operator coverage and integrations vary by device; adds a separate mobile release and test matrix. | Consider when the team is PyTorch-centric and target-device delegate support is proven. |
| **LiteRT** | Runs converted models using Google's on-device runtime and available Android/mobile delegates. | Strong Android ecosystem and optimized on-device execution paths; supports offline use. | Conversion and platform coverage differ from ONNX; quantization and delegate behavior require device-specific tests. | Attractive for an Android-first release where supported devices show a meaningful size, speed, or power gain. |
| **Core ML (iOS)** | Converts/loads a compatible model into Core ML and uses Apple CPU/GPU/Neural Engine scheduling. | Native iOS integration and access to Apple hardware acceleration. | Apple-only deployment and conversion/parity maintenance alongside Android. | A focused iOS optimization after the cross-platform model and clinical checks are established. |

**1. What is the trade-off between ONNX Runtime Mobile and LiteRT?**

ONNX Runtime Mobile favors a shared ONNX artifact and cross-platform engineering workflow; LiteRT can provide tighter Android integration and delegates, but may require a separate conversion, operator validation, and Android-focused maintenance. The smaller/faster choice must be established on the actual supported phones, not assumed from framework choice alone.

**2. Which options support fully offline use in rural clinics?**

All four can support local inference when the runtime, model, and required assets are packaged on-device; no network is needed for each prediction. The app still needs a secure, reliable process for initial installation and signed model updates, and must handle stale versions when devices are offline for long periods.

**3. What is likely to be most power efficient, and what is the trade-off?**

A runtime that successfully delegates to the device's Neural Engine, GPU, or other low-power accelerator (for example, Core ML on supported Apple devices or LiteRT delegates on supported Android devices) is a strong candidate. Delegate coverage is hardware-specific and can change numerical behavior; CPU fallback, energy use, thermal throttling, and sensitivity must all be measured on representative devices.

#### Strategic choice

**Recommendation for UdaciMed's initial mobile release:** Start with ONNX Runtime Mobile to reduce duplicated Android/iOS implementation and reach more regions with one model pipeline, while keeping inference fully offline. Treat this as a pilot, not automatic clearance for diagnosis: freeze the model and threshold, validate sensitivity and prediction parity on representative device/runtime combinations, document dataset and subgroup performance, and complete applicable regulatory/privacy review before clinical deployment. If profiling shows unacceptable battery, latency, or package size, add LiteRT for Android or Core ML for iOS as separately versioned backends, each with its own regression and clinical revalidation. This staged approach limits initial engineering and validation burden while retaining a path to platform-specific acceleration and global reach.

-----

## **Congratulations!**

You have successfully implemented a complete hardware-accelerated deployment pipeline! Let's recap the decisions you have made and results you have achieved while transforming an optimized model into a production-ready healthcare solution.

### **TODO: Production deployment scorecard**

**Final GPU deployment performance vs UdaciMed targets:**

_<\<Complete final scorecard based on your benchmarking results:>>_

| Metric | Target | Achieved | Status |
|--------|--------|----------|--------|
| **Memory Usage** | <100MB | | |
| **Throughput** | >2,000 samples/sec | | |
| **Latency** | <3ms | | | |
| **FLOP Reduction** | <0.4 GFLOPs per sample | | | |
| **Clinical Safety** | >98% sensitivity | | | |

_<\<Give yourself a final production score given the number of targets met>>_

**Overall production score: X/5 targets met!**

### **TODO: Strategic deployment insights**

_<\<Reflect on the key decisions you made, and why>>_

#### Mixed Precision Strategy
**Your FP16/FP32 choice:** # _(FP32, FP16)_

**Why you made this decision:**

#### Backend Selection
**Your ONNX execution provider choice:**  _(CPU EP, CUDA EP TensorRT EP, etc.)_

**Why this backend aligned with UdaciMed's requirements:**

#### Batching Configuration
**Your dynamic batching setup:** # _(preferred batch sizes, queue delay, etc.)_

**How this supports diverse clinical deployments:** 

### Optimization Philosophy
**Meeting targets vs maximizing metrics:**

_<\<What did you learn about when to stop optimizing and why?>>_

---

**You have completed the full journey from architectural optimization to production-ready deployment, demonstrating the technical skills and strategic thinking essential for deploying AI in healthcare. Your UdaciMed pneumonia detection system is now ready to serve hospitals worldwide while maintaining the clinical safety standards that save lives.**